# Processing

Construction du dataset final à partir des fichiers `processed` :

1. Suppression des agrégats géographiques en double (`China`)
2. Harmonisation des noms de pays du fichier de températures avec la nomenclature FAO
3. Pesticides issus du fichier FAO brut, joints sur le code M49 (quantité totale + intensité en kg/ha)
4. Fusion et contrôles d'intégrité

In [1]:
import pandas as pd
import os

## Récupération des fichiers processed

In [2]:
productivity = pd.read_csv('../data/processed/yield.csv', sep=',')
temp = pd.read_csv('../data/processed/temp.csv', sep=',')
rainfall = pd.read_csv('../data/processed/rainfall.csv', sep=',')
n_initial = len(productivity)
productivity.shape, temp.shape, rainfall.shape

((56717, 7), (28514, 3), (16770, 4))

## Agrégats géographiques en double

La FAO fournit à la fois `China` (agrégat, M49 159) et ses composantes `China, mainland`, `China, Taiwan Province of` et `China, Hong Kong SAR`, sur toute la période 1961-2016. Garder les deux reviendrait à compter deux fois les mêmes parcelles.

On conserve les composantes, plus fines et cohérentes avec le reste du dataset (un pays = une entité), et on supprime l'agrégat.

Les entités historiques (`USSR`, `Czechoslovakia`, `Yugoslav SFR`, `Sudan (former)`, `Belgium-Luxembourg`, `Ethiopia PDR`, `Serbia and Montenegro`) ne posent pas ce problème : elles se succèdent dans le temps sans chevauchement avec les États qui leur succèdent.

In [3]:
AGGREGATES = ['China']
productivity = productivity[~productivity['Area'].isin(AGGREGATES)].reset_index(drop=True)
print(f"{n_initial - len(productivity)} lignes d'agrégats supprimées")
n_initial = len(productivity)

448 lignes d'agrégats supprimées


In [4]:
# Contrôle : aucun chevauchement temporel pour les entités historiques
successions = {
    'USSR': ['Russian Federation'],
    'Czechoslovakia': ['Czechia', 'Slovakia'],
    'Yugoslav SFR': ['Serbia and Montenegro'],
    'Serbia and Montenegro': ['Serbia'],
    'Sudan (former)': ['Sudan', 'South Sudan'],
    'Belgium-Luxembourg': ['Belgium', 'Luxembourg'],
    'Ethiopia PDR': ['Ethiopia'],
}
years = productivity.groupby('Area')['Year'].agg(['min', 'max'])
for old, news in successions.items():
    for new in news:
        assert years.loc[old, 'max'] < years.loc[new, 'min'], f'Chevauchement {old} / {new}'
print('Aucun chevauchement temporel')

Aucun chevauchement temporel


## Harmonisation des noms de pays (températures)

Le fichier de températures n'a pas de code pays : la jointure se fait sur le nom. Plusieurs noms diffèrent de la nomenclature FAO (`United States` / `United States of America`, `Russia` / `Russian Federation`…), ce qui faisait perdre la température de pays majeurs. On les renomme avant la fusion.

In [5]:
TEMP_NAME_MAP = {
    'Bolivia': 'Bolivia (Plurinational State of)',
    'Bosnia And Herzegovina': 'Bosnia and Herzegovina',
    'China': 'China, mainland',
    'Congo (Democratic Republic Of The)': 'Democratic Republic of the Congo',
    'Czech Republic': 'Czechia',
    "Côte D'Ivoire": "Côte d'Ivoire",
    'Guinea Bissau': 'Guinea-Bissau',
    'Hong Kong': 'China, Hong Kong SAR',
    'Iran': 'Iran (Islamic Republic of)',
    'Laos': "Lao People's Democratic Republic",
    'Macedonia': 'The former Yugoslav Republic of Macedonia',
    'Moldova': 'Republic of Moldova',
    'Russia': 'Russian Federation',
    'South Korea': 'Republic of Korea',
    'Syria': 'Syrian Arab Republic',
    'Taiwan': 'China, Taiwan Province of',
    'Tanzania': 'United Republic of Tanzania',
    'United States': 'United States of America',
    'Venezuela': 'Venezuela (Bolivarian Republic of)',
    'Vietnam': 'Viet Nam',
}
temp['Area'] = temp['Area'].replace(TEMP_NAME_MAP)

# Sécurité : un renommage ne doit pas créer de doublon (Area, Year)
temp = temp.groupby(['Area', 'Year'], as_index=False)['avg_temp'].mean()

# Contrôle : tous les pays du fichier de températures doivent exister dans le dataset de rendement
unmatched = set(temp['Area']) - set(productivity['Area'])
assert not unmatched, f'Noms non appariés : {sorted(unmatched)}'
print(f"{temp['Area'].nunique()} pays avec température, tous appariés")

137 pays avec température, tous appariés


## Pesticides (fichier FAO brut)

Le fichier `Inputs_Pesticides_Use_E_All_Data_NOFLAG.csv` (FAOSTAT) remplace `pesticides.csv` :

- il contient le code **M49** de chaque pays, ce qui permet une jointure sur code plutôt que sur nom ;
- il couvre davantage de pays (Chine, Nigeria, Philippines, RDC…) ;
- il fournit l'**intensité d'usage** (`Use per area of cropland`, en kg/ha), plus pertinente que la quantité nationale totale, qui reflète surtout la taille du pays.

On conserve les deux : `pesticides_tonnes` (utilisée par les notebooks d'entraînement) et `pesticides_kg_ha`.

In [6]:
pesticides_raw = pd.read_csv('../data/raw/Inputs_Pesticides_Use_E_All_Data_NOFLAG.csv', encoding='latin-1')

ELEMENTS = {
    'Agricultural Use': 'pesticides_tonnes',          # t
    'Use per area of cropland': 'pesticides_kg_ha',   # kg/ha
}
pesticides = pesticides_raw[
    (pesticides_raw['Item'] == 'Pesticides (total)')
    & (pesticides_raw['Element'].isin(ELEMENTS))
    & (pesticides_raw['Area Code'] < 5000)  # codes >= 5000 : régions (World, Africa...)
].copy()
pesticides['Area ISO Code'] = pesticides['Area Code (M49)'].str.strip("'").astype('int64')

year_cols = [c for c in pesticides.columns if c.startswith('Y') and c[1:].isdigit()]
pesticides = pesticides.melt(
    id_vars=['Area ISO Code', 'Element'], value_vars=year_cols,
    var_name='Year', value_name='value',
)
pesticides['Year'] = pesticides['Year'].str[1:].astype('int64')
pesticides = (
    pesticides.pivot_table(index=['Area ISO Code', 'Year'], columns='Element', values='value')
    .rename(columns=ELEMENTS)
    .reset_index()
)
pesticides.columns.name = None
pesticides.head()

,Area ISO Code,Year,pesticides_tonnes,pesticides_kg_ha
0,8,1990,121.0,0.17
1,8,1991,121.0,0.17
2,8,1992,121.0,0.17
3,8,1993,121.0,0.17
4,8,1994,201.0,0.29


In [7]:
assert not pesticides.duplicated(['Area ISO Code', 'Year']).any()
covered = productivity[productivity['Area ISO Code'].isin(pesticides['Area ISO Code'])]['Area'].nunique()
print(f"{covered} / {productivity['Area'].nunique()} pays du dataset couverts par les données pesticides")

198 / 211 pays du dataset couverts par les données pesticides


## Fusion des datasets

- précipitations : jointure sur le code ISO Alpha-3 + année
- pesticides : jointure sur le code M49 + année
- températures : jointure sur le nom harmonisé + année

In [8]:
dataset = productivity.merge(
    rainfall.drop(columns=['Area']), how='left',
    left_on=['Year', 'Area ISO Alpha-3 Code'], right_on=['Year', 'Code'],
)
dataset = dataset.merge(pesticides, how='left', on=['Area ISO Code', 'Year'])
dataset = dataset.merge(temp, how='left', on=['Area', 'Year'])

dataset.head()

,Area,Item,Year,hg/ha,Area ISO Code,Area ISO Alpha-2 Code,Area ISO Alpha-3 Code,Code,average_rain_fall_mm_per_year,pesticides_tonnes,pesticides_kg_ha,avg_temp
0,Afghanistan,Maize,1961,14000,4,AF,AFG,AFG,375.61630,NaN,NaN,14.23
1,Afghanistan,Maize,1962,14000,4,AF,AFG,AFG,297.93750,NaN,NaN,14.10
2,Afghanistan,Maize,1963,14260,4,AF,AFG,AFG,340.40457,NaN,NaN,15.01
3,Afghanistan,Maize,1964,14257,4,AF,AFG,AFG,381.36438,NaN,NaN,13.73
4,Afghanistan,Maize,1965,14400,4,AF,AFG,AFG,387.33570,NaN,NaN,13.90


In [9]:
# Taux de valeurs manquantes par variable explicative (dataset complet et période 1990-2013)
features = ['average_rain_fall_mm_per_year', 'pesticides_tonnes', 'pesticides_kg_ha', 'avg_temp']
pd.DataFrame({
    '1961-2016': dataset[features].isnull().mean(),
    '1990-2013': dataset.loc[dataset['Year'].between(1990, 2013), features].isnull().mean(),
}).style.format('{:.1%}')

,1961-2016,1990-2013
average_rain_fall_mm_per_year,7.5%,7.0%
pesticides_tonnes,50.7%,4.8%
pesticides_kg_ha,51.1%,5.6%
avg_temp,32.8%,27.8%


In [10]:
dataset['missing_pesticide_value'] = dataset['pesticides_tonnes'].isnull().astype(int)
dataset['missing_rain_value'] = dataset['average_rain_fall_mm_per_year'].isnull().astype(int)
dataset['missing_temp_value'] = dataset['avg_temp'].isnull().astype(int)

In [12]:
dataset['Item'].value_counts()

Item
Maize                   8575
Potatoes                7820
Rice, paddy             6413
Sweet potatoes          6300
Wheat                   6104
Cassava                 5662
Sorghum                 5455
Soybeans                4136
Yams                    3150
Plantains and others    2654
Name: count, dtype: int64

In [13]:
localization_fr_name = {
    'Maize': 'Maïs',
    'Potatoes': 'Pomme de terre',
    'Rice, paddy': 'Riz',
    'Sweet potatoes': 'Patate douce',
    'Wheat': 'Blé',
    'Cassava': 'Manioc',
    'Sorghum': 'Sorgho',
    'Soybeans': 'Soja',
    'Yams': 'Igname',
    'Plantains and others': 'Plantains et autres'
}

dataset['Item'] = dataset['Item'].map(localization_fr_name)

## Sauvegarde du dataset final dans le dossier rafined

In [14]:
dataset_1990_2013 = dataset[(dataset['Year'] >= 1990) & (dataset['Year'] <= 2013)]
# Sauvegarde du dataset final dans le dossier processed
import os

if not os.path.exists('../data/rafined'):
    os.makedirs('../data/rafined')

dataset_1990_2013.to_csv('../data/rafined/dataset.csv', index=False)